<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes/blob/main/Script_principal_de_ejecuci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
main.py

"""

from matriz_carga import (
    validar_matrices,
    calcular_ocupacion,
    evaluar_balance,
    extraer_submatriz_critica,
)


def imprimir_matriz(titulo: str, matriz, unidad: str = "") -> None:
    """Imprime una matriz numérica con formato tabular legible."""
    print(f"\n{titulo}")
    print("-" * len(titulo))
    for fila in matriz:
        celdas = [f"{valor:8.2f}{unidad}" for valor in fila]
        print("  ".join(celdas))


def main() -> None:
    # -----------------------------------------------------------------
    # Datos de prueba: bahía de carga simulada de 4 filas x 5 columnas
    # -----------------------------------------------------------------
    cargas_reales = [
        [180, 220, 300, 150, 90],
        [400, 310, 280, 200, 175],
        [90,  120, 500, 260, 130],
        [210, 150, 170, 190, 220],
    ]

    capacidades_maximas = [
        [200, 250, 300, 200, 100],
        [350, 300, 300, 250, 200],
        [100, 150, 400, 300, 150],
        [250, 200, 200, 200, 250],
    ]

    tolerancia_desbalance_kg = 50.0
    ventana_k, ventana_p = 2, 2

    print("=" * 60)
    print(" AUDITORÍA Y BALANCE MATRICIAL DE CARGA - AeroCargo-Matrix ")
    print("=" * 60)

    # 1. Validación dimensional
    es_valida = validar_matrices(cargas_reales, capacidades_maximas)
    print(f"\n[1] Validación de matrices de entrada: {'APROBADA' if es_valida else 'RECHAZADA'}")

    if not es_valida:
        print("Las matrices de entrada no son válidas. Terminando ejecución.")
        return

    # 2. Ocupación y sobrecarga
    resultado_ocupacion = calcular_ocupacion(cargas_reales, capacidades_maximas)
    porcentajes = resultado_ocupacion["porcentajes"]
    sobrecargas = resultado_ocupacion["sobrecargas"]

    imprimir_matriz("[2] Matriz de Porcentaje de Ocupación (%)", porcentajes, "%")

    print(f"\nCeldas en sobrecarga crítica (> 100.0%): {len(sobrecargas)}")
    for (i, j) in sobrecargas:
        print(f"  - Celda ({i}, {j}): {porcentajes[i][j]:.2f}% "
              f"[{cargas_reales[i][j]} kg / {capacidades_maximas[i][j]} kg]")

    # 3. Balance lateral y longitudinal
    resultado_balance = evaluar_balance(cargas_reales, tolerancia_desbalance_kg)

    print("\n[3] Balance de Carga")
    print("-" * 20)
    print("Peso total por fila longitudinal (kg):")
    for i, peso in enumerate(resultado_balance["pesos_fila"]):
        print(f"  Fila {i}: {peso:.2f} kg")

    print(f"\nDesbalance lateral: {resultado_balance['desbalance_lateral']:.2f} kg "
          f"(tolerancia: {tolerancia_desbalance_kg:.2f} kg)")
    estado_balance = "DENTRO DE TOLERANCIA" if resultado_balance["balance_ok"] else "FUERA DE TOLERANCIA"
    print(f"Estado de balance: {estado_balance}")

    # 4. Submatriz de sobrecarga crítica
    submatriz_critica = extraer_submatriz_critica(porcentajes, ventana_k, ventana_p)
    imprimir_matriz(
        f"[4] Submatriz Crítica Detectada ({ventana_k}x{ventana_p})",
        submatriz_critica,
        "%",
    )

    # -----------------------------------------------------------------
    # Caso de borde adicional: matriz mínima válida 2x2
    # -----------------------------------------------------------------
    print("\n" + "=" * 60)
    print(" DEMOSTRACIÓN DE CASO DE BORDE: MATRIZ MÍNIMA 2x2 ")
    print("=" * 60)

    cargas_min = [[0, 100], [50, 200]]
    capacidades_min = [[100, 100], [100, 200]]

    print(f"\n[1] Validación matriz 2x2: "
          f"{'APROBADA' if validar_matrices(cargas_min, capacidades_min) else 'RECHAZADA'}")

    resultado_min = calcular_ocupacion(cargas_min, capacidades_min)
    imprimir_matriz("[2] Ocupación matriz 2x2 (%)", resultado_min["porcentajes"], "%")

    resultado_balance_min = evaluar_balance(cargas_min, tolerancia_desbalance_kg)
    print(f"\n[3] Desbalance lateral (2x2, M par): "
          f"{resultado_balance_min['desbalance_lateral']:.2f} kg")


if __name__ == "__main__":
    main()